# 🛒 Shopper Spectrum: Customer Segmentation & Product Recommendation

**Project:** Customer Segmentation and Product Recommendation in E-Commerce

**Domain:** E-Commerce and Retail Analytics

**Skills:** Data Cleaning, EDA, RFM Analysis, KMeans Clustering, Collaborative Filtering, Streamlit

---

## 📋 Table of Contents

1. [Environment Setup](#step-1)
2. [Data Preprocessing](#step-2)
3. [Exploratory Data Analysis (EDA)](#step-3)
4. [RFM Analysis & Customer Segmentation](#step-4)
5. [Product Recommendation System](#step-5)
6. [Streamlit Application](#step-6)

---

**Dataset:** `online_retail.csv` — Online retail transaction records (Dec 2022 – Dec 2023)

**Columns:** InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country

## 🚀 Step 1: Environment Setup

### Folder Structure
```
shopper_spectrum/
├── data/
│   └── online_retail.csv
├── notebooks/
│   └── shopper_spectrum.ipynb  ← This file
├── models/
├── app/
└── outputs/
    └── eda_charts/
```

### Virtual Environment & Packages
Run these commands in your terminal (CMD/PowerShell):

```bash
cd C:\Users\steph\Desktop\shopper_spectrum
python -m venv venv
venv\Scripts\activate
pip install pandas numpy matplotlib seaborn scikit-learn streamlit plotly joblib jupyter notebook ipykernel
```

### Import Libraries
Run the cell below to import all required libraries.

In [ ]:
# ============================================================
# STEP 1: IMPORT ALL LIBRARIES
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# Set visual style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

# Auto-detect project root (works whether running from notebooks/ or root)
notebook_dir = os.path.abspath('.')
project_root = os.path.dirname(notebook_dir) if os.path.basename(notebook_dir) == 'notebooks' else notebook_dir
os.chdir(project_root)
print(f"📁 Project root: {os.getcwd()}")

# Create output directories
os.makedirs('outputs/eda_charts', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('app', exist_ok=True)

print("✅ All libraries imported and directories created!")

---

## 🔧 Step 2: Data Preprocessing

### Objective
Clean the raw `online_retail.csv` dataset to ensure reliable analysis.

### Cleaning Rules (from Project Requirements)
| # | Rule | Reason |
|---|------|--------|
| 1 | Remove missing `CustomerID` | Can't analyze unknown customers |
| 2 | Exclude cancelled invoices (starting with 'C') | Returns, not purchases |
| 3 | Remove negative/zero `Quantity` | Invalid transactions |
| 4 | Remove negative/zero `UnitPrice` | Data entry errors |
| 5 | Remove non-product StockCodes | POST, DOT, D, C2, M, BANK CHARGES |
| 6 | Remove blank `Description` | Can't identify products |
| 7 | Remove duplicates | Exact duplicate records |
| 8 | Create `TotalAmount` | Quantity × UnitPrice |

### Load & Clean Data

In [ ]:
# ============================================================
# STEP 2: DATA PREPROCESSING
# ============================================================
print("=" * 70)
print("🔧 STEP 2: DATA PREPROCESSING")
print("=" * 70)

# 2.1 Load dataset
print("\n📥 Loading dataset...")
df = pd.read_csv('data/online_retail.csv', encoding='latin1')
original_shape = df.shape
print(f"✅ Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

# 2.2 Parse dates
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], format='%d-%m-%Y %H:%M')

# 2.3 Initial data check
print("\n🔍 Initial data check:")
print(f"   Missing CustomerID: {df['CustomerID'].isnull().sum():,}")
print(f"   Cancelled invoices: {df['InvoiceNo'].astype(str).str.startswith('C').sum():,}")
print(f"   Negative quantities: {(df['Quantity'] <= 0).sum():,}")
print(f"   Negative/zero prices: {(df['UnitPrice'] <= 0).sum():,}")
print(f"   Blank descriptions: {df['Description'].isnull().sum():,}")
print(f"   Duplicate rows: {df.duplicated().sum():,}")

# 2.4 Apply cleaning pipeline
df_clean = df.copy()

# Remove missing CustomerID
df_clean = df_clean.dropna(subset=['CustomerID'])

# Remove cancelled invoices
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]

# Remove negative/zero quantities
df_clean = df_clean[df_clean['Quantity'] > 0]

# Remove negative/zero prices
df_clean = df_clean[df_clean['UnitPrice'] > 0]

# Remove non-product StockCodes
non_product_codes = ['POST', 'DOT', 'D', 'C2', 'M', 'BANK CHARGES', 'CRUK', 'PADS']
df_clean = df_clean[~df_clean['StockCode'].isin(non_product_codes)]

# Remove blank descriptions
df_clean = df_clean.dropna(subset=['Description'])
df_clean = df_clean[df_clean['Description'].str.strip() != '']

# Remove duplicates
df_clean = df_clean.drop_duplicates()

# Convert CustomerID to integer
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)

# Create TotalAmount
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['UnitPrice']

# 2.5 Cleaning summary
print("\n" + "=" * 70)
print("📊 CLEANING SUMMARY")
print("=" * 70)
print(f"   Original rows:      {original_shape[0]:>10,}")
print(f"   Final rows:           {len(df_clean):>10,}")
print(f"   Rows removed:         {original_shape[0] - len(df_clean):>10,}")
print(f"   Retention rate:       {len(df_clean)/original_shape[0]*100:>9.2f}%")

# 2.6 Verification
print("\n✅ Verification:")
print(f"   Missing CustomerID: {df_clean['CustomerID'].isnull().sum()}")
print(f"   Cancelled invoices: {df_clean['InvoiceNo'].astype(str).str.startswith('C').sum()}")
print(f"   Negative quantities: {(df_clean['Quantity'] <= 0).sum()}")
print(f"   Negative prices: {(df_clean['UnitPrice'] <= 0).sum()}")
print(f"   Unique customers: {df_clean['CustomerID'].nunique():,}")
print(f"   Unique products: {df_clean['StockCode'].nunique():,}")
print(f"   Date range: {df_clean['InvoiceDate'].min()} to {df_clean['InvoiceDate'].max()}")

# 2.7 Save cleaned data
df_clean.to_csv('data/online_retail_cleaned.csv', index=False)
print("\n💾 Cleaned data saved to: data/online_retail_cleaned.csv")
print("\n🎉 STEP 2 COMPLETE!")

---

## 📊 Step 3: Exploratory Data Analysis (EDA)

### Objectives
- Understand transaction volume by country
- Identify top-selling products
- Visualize purchase trends over time
- Inspect monetary distribution per transaction and customer
- Analyze customer activity patterns

### Charts Generated
1. Country-wise revenue bar chart
2. Country revenue pie chart
3. Top products by quantity
4. Top products by revenue
5. Monthly revenue trend
6. Monthly orders trend
7. Daily revenue with 7-day moving average
8. Transaction value distribution
9. Customer lifetime value distribution
10. Top 10 customers by spend
11. Quantity & price distributions
12. Sales by day-of-week & hour

In [ ]:
# ============================================================
# STEP 3: EXPLORATORY DATA ANALYSIS (EDA)
# ============================================================
print("=" * 70)
print("📊 STEP 3: EXPLORATORY DATA ANALYSIS")
print("=" * 70)

# Load cleaned data
df = pd.read_csv('data/online_retail_cleaned.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f"\n✅ Loaded: {df.shape[0]:,} rows")

# --- 3.1 Country-wise Sales ---
country_sales = df.groupby('Country').agg({
    'TotalAmount': 'sum',
    'InvoiceNo': 'nunique',
    'CustomerID': 'nunique'
}).round(2)
country_sales.columns = ['TotalRevenue', 'NumOrders', 'NumCustomers']
country_sales = country_sales.sort_values('TotalRevenue', ascending=False)
country_sales.to_csv('outputs/country_sales_summary.csv')
print("\n📊 Top 5 Countries by Revenue:")
print(country_sales.head())

# Chart 1: Top 10 Countries by Revenue
fig, ax = plt.subplots(figsize=(12, 6))
top10 = country_sales.head(10)
bars = ax.barh(range(len(top10)), top10['TotalRevenue'], color='steelblue')
ax.set_yticks(range(len(top10)))
ax.set_yticklabels(top10.index)
ax.invert_yaxis()
ax.set_xlabel('Total Revenue (£)')
ax.set_title('Top 10 Countries by Total Revenue', fontsize=14, fontweight='bold')
for i, bar in enumerate(bars):
    width = bar.get_width()
    ax.text(width + width*0.01, bar.get_y() + bar.get_height()/2,
            f'£{width:,.0f}', ha='left', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('outputs/eda_charts/01_country_revenue.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart 1 saved")

# --- 3.2 Top Selling Products ---
product_summary = df.groupby('Description').agg({
    'Quantity': 'sum',
    'TotalAmount': 'sum',
    'InvoiceNo': 'nunique'
}).round(2)
product_summary.columns = ['TotalQuantity', 'TotalRevenue', 'NumOrders']
product_summary.to_csv('outputs/product_summary.csv')

# Chart 3: Top 10 by Quantity
fig, ax = plt.subplots(figsize=(12, 7))
top10_qty = product_summary.sort_values('TotalQuantity', ascending=False).head(10)
bars = ax.barh(range(len(top10_qty)), top10_qty['TotalQuantity'], color='coral')
ax.set_yticks(range(len(top10_qty)))
ax.set_yticklabels([d[:40] + '...' if len(d) > 40 else d for d in top10_qty.index])
ax.invert_yaxis()
ax.set_xlabel('Total Quantity Sold')
ax.set_title('Top 10 Products by Quantity Sold', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/eda_charts/03_top_products_quantity.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart 3 saved")

# Chart 4: Top 10 by Revenue
fig, ax = plt.subplots(figsize=(12, 7))
top10_rev = product_summary.sort_values('TotalRevenue', ascending=False).head(10)
bars = ax.barh(range(len(top10_rev)), top10_rev['TotalRevenue'], color='mediumseagreen')
ax.set_yticks(range(len(top10_rev)))
ax.set_yticklabels([d[:40] + '...' if len(d) > 40 else d for d in top10_rev.index])
ax.invert_yaxis()
ax.set_xlabel('Total Revenue (£)')
ax.set_title('Top 10 Products by Revenue', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/eda_charts/04_top_products_revenue.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart 4 saved")

# --- 3.3 Monthly Trends ---
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')
monthly = df.groupby('YearMonth').agg({
    'TotalAmount': 'sum',
    'InvoiceNo': 'nunique',
    'CustomerID': 'nunique'
}).round(2)
monthly.columns = ['Revenue', 'Orders', 'Customers']
monthly.to_csv('outputs/monthly_sales_summary.csv')

# Chart 5: Monthly Revenue Trend
fig, ax = plt.subplots(figsize=(14, 6))
months = [str(m) for m in monthly.index]
ax.plot(months, monthly['Revenue'], marker='o', linewidth=2.5, color='navy')
ax.fill_between(months, monthly['Revenue'], alpha=0.3, color='lightblue')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (£)')
ax.set_title('Monthly Revenue Trend', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('outputs/eda_charts/05_monthly_revenue_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart 5 saved")

# --- 3.4 Customer Spending ---
customer_stats = df.groupby('CustomerID').agg({
    'TotalAmount': 'sum',
    'InvoiceNo': 'nunique',
    'Quantity': 'sum'
}).round(2)
customer_stats.columns = ['TotalSpend', 'NumOrders', 'TotalQuantity']
customer_stats.to_csv('outputs/customer_stats_summary.csv')

# Chart 9: Customer Spend Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(customer_stats['TotalSpend'], bins=50, color='gold', edgecolor='black', alpha=0.7)
axes[0].axvline(customer_stats['TotalSpend'].mean(), color='red', linestyle='--', label=f"Mean: £{customer_stats['TotalSpend'].mean():.2f}")
axes[0].axvline(customer_stats['TotalSpend'].median(), color='green', linestyle='--', label=f"Median: £{customer_stats['TotalSpend'].median():.2f}")
axes[0].set_xlabel('Total Customer Spend (£)')
axes[0].set_ylabel('Number of Customers')
axes[0].set_title('Distribution of Customer Lifetime Value')
axes[0].legend()
axes[1].boxplot(customer_stats['TotalSpend'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='plum', alpha=0.7))
axes[1].set_ylabel('Total Customer Spend (£)')
axes[1].set_title('Customer Spend Box Plot')
plt.tight_layout()
plt.savefig('outputs/eda_charts/09_customer_spend_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart 9 saved")

# Chart 10: Top 10 Customers
fig, ax = plt.subplots(figsize=(12, 6))
top10_cust = customer_stats.sort_values('TotalSpend', ascending=False).head(10)
bars = ax.barh(range(len(top10_cust)), top10_cust['TotalSpend'], color='teal')
ax.set_yticks(range(len(top10_cust)))
ax.set_yticklabels([f'Customer {int(cid)}' for cid in top10_cust.index])
ax.invert_yaxis()
ax.set_xlabel('Total Spend (£)')
ax.set_title('Top 10 Customers by Total Spend', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/eda_charts/10_top_customers.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart 10 saved")

print("\n🎉 STEP 3 COMPLETE! All charts saved to outputs/eda_charts/")

---

## 🎯 Step 4: RFM Analysis & Customer Segmentation

### What is RFM?
- **Recency (R):** How recently did the customer purchase? *(Lower = better)*
- **Frequency (F):** How often do they purchase? *(Higher = better)*
- **Monetary (M):** How much do they spend? *(Higher = better)*

### Methodology
1. Calculate RFM values per customer
2. Log-transform skewed features (Frequency, Monetary)
3. Standardize using `StandardScaler`
4. Use Elbow Method + Silhouette Score to find optimal k
5. Run KMeans clustering (k=4)
6. Auto-label clusters: **High-Value, Regular, Occasional, At-Risk**

### Cluster Labeling Rules
| Segment | Characteristics |
|---------|-------------------|
| 🟢 High-Value | Low R, High F, High M |
| 🔵 Regular | Medium F, Medium M |
| 🟡 Occasional | Low F, Low M, High R |
| 🔴 At-Risk | High R, Low F, Low M |

In [ ]:
# ============================================================
# STEP 4: RFM ANALYSIS & CUSTOMER SEGMENTATION
# ============================================================
print("=" * 70)
print("🎯 STEP 4: RFM ANALYSIS & CUSTOMER SEGMENTATION")
print("=" * 70)

# Load cleaned data
df = pd.read_csv('data/online_retail_cleaned.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# 4.1 Calculate RFM values
reference_date = df['InvoiceDate'].max() + timedelta(days=1)
print(f"\n📅 Reference date: {reference_date.date()}")

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (reference_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalAmount': 'sum'
}).round(2)
rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm = rfm.reset_index()

print(f"\n📋 RFM Table: {rfm.shape[0]:,} customers")
print(rfm[['Recency', 'Frequency', 'Monetary']].describe())

# 4.2 RFM Distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].hist(rfm['Recency'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Recency Distribution')
axes[0].set_xlabel('Days Since Last Purchase')
axes[1].hist(rfm['Frequency'], bins=50, color='coral', edgecolor='black', alpha=0.7)
axes[1].set_title('Frequency Distribution')
axes[1].set_xlabel('Number of Transactions')
axes[2].hist(rfm['Monetary'], bins=50, color='mediumseagreen', edgecolor='black', alpha=0.7)
axes[2].set_title('Monetary Distribution')
axes[2].set_xlabel('Total Spend (£)')
plt.tight_layout()
plt.savefig('outputs/eda_charts/13_rfm_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ RFM distributions saved")

# 4.3 RFM Correlation Heatmap
fig, ax = plt.subplots(figsize=(8, 6))
corr = rfm[['Recency', 'Frequency', 'Monetary']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.3f', square=True, ax=ax)
ax.set_title('RFM Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/eda_charts/14_rfm_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Correlation heatmap saved")

# 4.4 Log transform + Standardize
rfm_log = rfm.copy()
rfm_log['Frequency_log'] = np.log1p(rfm_log['Frequency'])
rfm_log['Monetary_log'] = np.log1p(rfm_log['Monetary'])

features = ['Recency', 'Frequency_log', 'Monetary_log']
X = rfm_log[features].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\n⚖️  Standardization complete")
print(f"   Mean: {X_scaled.mean(axis=0).round(4)}")
print(f"   Std:  {X_scaled.std(axis=0).round(4)}")

# 4.5 Elbow Method + Silhouette Score
inertias = []
silhouettes = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))

# Elbow plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(K_range, inertias, marker='o', linewidth=2.5, color='navy')
ax.axvline(x=4, color='red', linestyle='--', label='k=4 (selected)')
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Inertia (WCSS)')
ax.set_title('Elbow Method for Optimal k', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/eda_charts/15_elbow_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Elbow curve saved")

# Silhouette plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(K_range, silhouettes, marker='s', linewidth=2.5, color='darkgreen')
ax.axvline(x=4, color='red', linestyle='--', label='k=4 (selected)')
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette Score vs k', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/eda_charts/16_silhouette_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Silhouette scores saved")

# 4.6 Run KMeans (k=4)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm_log['Cluster'] = kmeans.fit_predict(X_scaled)
rfm['Cluster'] = rfm_log['Cluster']

print("\n🤖 KMeans clustering complete")
print(f"   Cluster distribution:")
print(rfm['Cluster'].value_counts().sort_index())

# 4.7 Auto-label clusters
profile = rfm.groupby('Cluster').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': 'mean',
    'CustomerID': 'count'
}).reset_index()

profile['R_score'] = 1 - (profile['Recency'] - profile['Recency'].min()) / (profile['Recency'].max() - profile['Recency'].min() + 1e-9)
profile['F_score'] = (profile['Frequency'] - profile['Frequency'].min()) / (profile['Frequency'].max() - profile['Frequency'].min() + 1e-9)
profile['M_score'] = (profile['Monetary'] - profile['Monetary'].min()) / (profile['Monetary'].max() - profile['Monetary'].min() + 1e-9)
profile['Composite'] = profile['R_score'] + profile['F_score'] + profile['M_score']
profile_sorted = profile.sort_values('Composite', ascending=False)

clusters = profile_sorted['Cluster'].tolist()
labels_map = {}
labels_map[clusters[0]] = 'High-Value'
labels_map[clusters[3]] = 'At-Risk'

mid1, mid2 = clusters[1], clusters[2]
if profile[profile['Cluster'] == mid1]['Frequency'].values[0] > profile[profile['Cluster'] == mid2]['Frequency'].values[0]:
    labels_map[mid1] = 'Regular'
    labels_map[mid2] = 'Occasional'
else:
    labels_map[mid1] = 'Occasional'
    labels_map[mid2] = 'Regular'

rfm['Segment'] = rfm['Cluster'].map(labels_map)
rfm_log['Segment'] = rfm_log['Cluster'].map(labels_map)

print("\n🏷️  Segment Mapping:")
for cid, label in sorted(labels_map.items()):
    count = rfm[rfm['Cluster'] == cid].shape[0]
    print(f"   Cluster {cid} → {label:12s} ({count:,} customers)")

# 4.8 Cluster profile visualization
segment_order = ['High-Value', 'Regular', 'Occasional', 'At-Risk']
segment_colors = {'High-Value': '#2ecc71', 'Regular': '#3498db', 'Occasional': '#f39c12', 'At-Risk': '#e74c3c'}
segment_means = rfm.groupby('Segment')[['Recency', 'Frequency', 'Monetary']].mean().reindex(segment_order)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
segment_means['Recency'].plot(kind='bar', ax=axes[0], color=[segment_colors[s] for s in segment_order], rot=0)
axes[0].set_title('Average Recency by Segment')
axes[0].set_ylabel('Days')
segment_means['Frequency'].plot(kind='bar', ax=axes[1], color=[segment_colors[s] for s in segment_order], rot=0)
axes[1].set_title('Average Frequency by Segment')
axes[1].set_ylabel('Transactions')
segment_means['Monetary'].plot(kind='bar', ax=axes[2], color=[segment_colors[s] for s in segment_order], rot=0)
axes[2].set_title('Average Monetary by Segment')
axes[2].set_ylabel('Total Spend (£)')
plt.tight_layout()
plt.savefig('outputs/eda_charts/17_cluster_profiles.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Cluster profiles saved")

# 4.9 3D Cluster Visualization
fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111, projection='3d')
for segment in segment_order:
    subset = rfm[rfm['Segment'] == segment]
    ax.scatter(subset['Recency'], subset['Frequency'], subset['Monetary'],
               c=segment_colors[segment], label=segment, s=30, alpha=0.6)
ax.set_xlabel('Recency (days)')
ax.set_ylabel('Frequency (orders)')
ax.set_zlabel('Monetary (£)')
ax.set_title('3D Customer Segments (RFM)', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/eda_charts/18_3d_clusters.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ 3D clusters saved")

# 4.10 Save outputs
rfm.to_csv('outputs/rfm_table.csv', index=False)
joblib.dump(kmeans, 'models/kmeans_model.pkl')
joblib.dump(scaler, 'models/scaler.pkl')

cluster_profiles = rfm.groupby('Segment').agg({
    'Recency': ['mean', 'min', 'max'],
    'Frequency': ['mean', 'min', 'max'],
    'Monetary': ['mean', 'min', 'max'],
    'CustomerID': 'count'
}).round(2)
cluster_profiles.columns = ['R_Mean','R_Min','R_Max','F_Mean','F_Min','F_Max','M_Mean','M_Min','M_Max','Count']
cluster_profiles = cluster_profiles.reset_index()
cluster_profiles.to_csv('outputs/cluster_profiles.csv', index=False)

segment_mapping = pd.DataFrame([{'Cluster': k, 'Segment': v} for k, v in labels_map.items()])
segment_mapping.to_csv('outputs/segment_mapping.csv', index=False)

print("\n💾 Saved: outputs/rfm_table.csv")
print("💾 Saved: outputs/cluster_profiles.csv")
print("💾 Saved: outputs/segment_mapping.csv")
print("💾 Saved: models/kmeans_model.pkl")
print("💾 Saved: models/scaler.pkl")
print("\n🎉 STEP 4 COMPLETE!")

---

## 🛒 Step 5: Product Recommendation System

### Approach: Item-Based Collaborative Filtering

**Method:** Cosine Similarity on CustomerID × Description purchase matrix

**How it works:**
1. Create a matrix where rows = customers, columns = products, values = quantity purchased
2. Compute cosine similarity between every pair of products
3. For any given product, recommend the top 5 most similar products

**Why cosine similarity?** It measures the angle between two product vectors, capturing purchase pattern similarity regardless of absolute quantity.

In [ ]:
# ============================================================
# STEP 5: PRODUCT RECOMMENDATION SYSTEM
# ============================================================
print("=" * 70)
print("🛒 STEP 5: PRODUCT RECOMMENDATION SYSTEM")
print("=" * 70)

# Load cleaned data
df = pd.read_csv('data/online_retail_cleaned.csv')

# 5.1 Build user-item matrix
print("\n📊 Building CustomerID × Description matrix...")
user_item_matrix = df.pivot_table(
    index='CustomerID',
    columns='Description',
    values='Quantity',
    aggfunc='sum',
    fill_value=0
)
print(f"   Matrix shape: {user_item_matrix.shape}")
print(f"   Sparsity: {(user_item_matrix == 0).sum().sum() / (user_item_matrix.shape[0] * user_item_matrix.shape[1]) * 100:.2f}%")

# 5.2 Compute item-item cosine similarity
print("\n🔗 Computing cosine similarity...")
item_similarity = cosine_similarity(user_item_matrix.T)
similarity_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns
)
print(f"   Similarity matrix shape: {similarity_df.shape}")
print("   ✅ Cosine similarity computed")

# 5.3 Recommendation function
def get_recommendations(product_name, top_n=5):
    if product_name in similarity_df.index:
        matched = product_name
    else:
        matches = similarity_df.index[similarity_df.index.str.contains(product_name, case=False, na=False)]
        if len(matches) == 0:
            return None
        matched = matches[0]
    sim_scores = similarity_df[matched].drop(matched)
    top_products = sim_scores.sort_values(ascending=False).head(top_n)
    return [(prod, round(score, 4)) for prod, score in top_products.items()], matched

# 5.4 Test with popular products
popular = df.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(5)
print("\n🧪 Testing recommendations:")
for prod in popular.index[:3]:
    recs, matched = get_recommendations(prod, top_n=5)
    print(f"\n🎯 '{matched[:50]}...'")
    for i, (p, s) in enumerate(recs, 1):
        print(f"   {i}. {p[:50]}... (score: {s})")

# 5.5 Save models
product_list = sorted(similarity_df.index.tolist())
joblib.dump(similarity_df, 'models/product_similarity.pkl')
joblib.dump(product_list, 'models/product_names.pkl')

with open('outputs/product_list.txt', 'w', encoding='utf-8') as f:
    for p in product_list:
        f.write(p + "\n")

print("\n💾 Saved: models/product_similarity.pkl")
print(f"💾 Saved: models/product_names.pkl ({len(product_list):,} products)")
print("💾 Saved: outputs/product_list.txt")

# 5.6 Similarity heatmap (top 20 products)
top20 = popular.head(20).index.tolist()
sim_subset = similarity_df.loc[top20, top20]
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(sim_subset, cmap='coolwarm', center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Product Similarity Matrix (Top 20)', fontsize=14, fontweight='bold')
labels = [p[:25] + '...' if len(p) > 25 else p for p in top20]
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(labels, rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('outputs/eda_charts/19_product_similarity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Similarity heatmap saved")
print("\n🎉 STEP 5 COMPLETE!")

---

## 📱 Step 6: Streamlit Application

### Features

**Module 1: Product Recommendation**
- Text input for product name
- Fuzzy matching for partial names
- Returns top 5 similar products with similarity scores

**Module 2: Customer Segmentation**
- 3 number inputs: Recency (days), Frequency (purchases), Monetary (£)
- Predicts segment using saved KMeans model + scaler
- Color-coded result card with segment description

### How to Run
```bash
cd C:\Users\steph\Desktop\shopper_spectrum
venv\Scripts\activate
streamlit run app\app.py
```

The cell below will **automatically generate `app.py`** and **launch Streamlit** when you run it.

In [ ]:
# ============================================================
# STEP 6: GENERATE STREAMLIT APP + AUTO-LAUNCH
# ============================================================
print("=" * 70)
print("📱 STEP 6: STREAMLIT APP GENERATION & LAUNCH")
print("=" * 70)

streamlit_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import os

st.set_page_config(page_title="Shopper Spectrum", page_icon="🛒", layout="wide")

PAGES = ["🏠 Home", "🔍 Product Recommender", "👤 Customer Segmentation"]
if "current_page" not in st.session_state:
    st.session_state.current_page = "🏠 Home"

@st.cache_resource
def load_models():
    base_path = os.path.dirname(os.path.abspath(__file__))
    parent_path = os.path.dirname(base_path)
    models = {}
    models["kmeans"] = joblib.load(os.path.join(parent_path, "models", "kmeans_model.pkl")) if os.path.exists(os.path.join(parent_path, "models", "kmeans_model.pkl")) else None
    models["scaler"] = joblib.load(os.path.join(parent_path, "models", "scaler.pkl")) if os.path.exists(os.path.join(parent_path, "models", "scaler.pkl")) else None
    models["similarity_df"] = joblib.load(os.path.join(parent_path, "models", "product_similarity.pkl")) if os.path.exists(os.path.join(parent_path, "models", "product_similarity.pkl")) else None
    models["product_list"] = joblib.load(os.path.join(parent_path, "models", "product_names.pkl")) if os.path.exists(os.path.join(parent_path, "models", "product_names.pkl")) else None
    seg_path = os.path.join(parent_path, "outputs", "segment_mapping.csv")
    models["segment_mapping"] = pd.read_csv(seg_path) if os.path.exists(seg_path) else None
    return models

models = load_models()

def get_recommendations(product_name, top_n=5):
    similarity_df = models["similarity_df"]
    product_list = models["product_list"]
    if similarity_df is None or product_list is None:
        return None, "Models not loaded."
    if product_name in similarity_df.index:
        matched = product_name
    else:
        matches = [p for p in product_list if product_name.lower() in p.lower()]
        if not matches:
            return None, f"No product found matching '{product_name}'."
        matched = matches[0]
    sim_scores = similarity_df[matched].drop(matched)
    top_products = sim_scores.sort_values(ascending=False).head(top_n)
    return [(prod, round(score, 4)) for prod, score in top_products.items()], matched

def predict_segment(recency, frequency, monetary):
    kmeans = models["kmeans"]
    scaler = models["scaler"]
    seg_map = models["segment_mapping"]
    if kmeans is None or scaler is None or seg_map is None:
        return None, "Models not loaded."
    freq_log = np.log1p(frequency)
    mon_log = np.log1p(monetary)
    X = np.array([[recency, freq_log, mon_log]])
    X_scaled = scaler.transform(X)
    cluster = kmeans.predict(X_scaled)[0]
    segment = seg_map[seg_map["Cluster"] == cluster]["Segment"].values[0]
    return segment, cluster

with st.sidebar:
    st.title("🛒 Shopper Spectrum")
    st.markdown("---")
    selected_page = st.radio("Navigate to:", PAGES, index=PAGES.index(st.session_state.current_page))
    if selected_page != st.session_state.current_page:
        st.session_state.current_page = selected_page
        st.rerun()

page = st.session_state.current_page

if page == "🏠 Home":
    st.title("🛒 Welcome to Shopper Spectrum")
    col1, col2 = st.columns(2)
    with col1:
        st.subheader("🔍 Product Recommender")
        st.markdown("Enter a product name to get 5 similar recommendations.")
        if st.button("Go to Product Recommender →", key="btn_prod"):
            st.session_state.current_page = "🔍 Product Recommender"
            st.rerun()
    with col2:
        st.subheader("👤 Customer Segmentation")
        st.markdown("Enter RFM values to predict customer segment.")
        if st.button("Go to Customer Segmentation →", key="btn_seg"):
            st.session_state.current_page = "👤 Customer Segmentation"
            st.rerun()

elif page == "🔍 Product Recommender":
    st.title("🔍 Product Recommendation")
    if models["similarity_df"] is None:
        st.error("❌ Product similarity model not found.")
    else:
        product_input = st.text_input("Product Name", placeholder="e.g., WHITE HANGING HEART...")
        if st.button("🎯 Get Recommendations", type="primary"):
            if not product_input.strip():
                st.warning("Please enter a product name.")
            else:
                with st.spinner("Finding similar products..."):
                    recs, matched = get_recommendations(product_input.strip(), top_n=5)
                if recs is None:
                    st.error(f"❌ {matched}")
                else:
                    st.success(f"✅ Recommendations for: {matched}")
                    for i, (prod, score) in enumerate(recs, 1):
                        st.markdown(f"{i}. **{prod}** — Similarity: `{score}`")

elif page == "👤 Customer Segmentation":
    st.title("👤 Customer Segment Predictor")
    if models["kmeans"] is None or models["scaler"] is None:
        st.error("❌ Segmentation models not found.")
    else:
        col1, col2 = st.columns([1, 1.5])
        with col1:
            st.subheader("📊 Input RFM Values")
            recency = st.number_input("Recency (days)", min_value=0, max_value=1000, value=30)
            frequency = st.number_input("Frequency (purchases)", min_value=1, max_value=500, value=5)
            monetary = st.number_input("Monetary (£)", min_value=0.0, max_value=50000.0, value=500.0)
            if st.button("🎯 Predict Segment", type="primary"):
                with st.spinner("Analyzing..."):
                    segment, cluster = predict_segment(recency, frequency, monetary)
                if segment is None:
                    st.error(f"❌ {cluster}")
                else:
                    st.session_state["prediction"] = {"segment": segment, "cluster": cluster, "recency": recency, "frequency": frequency, "monetary": monetary}
        with col2:
            st.subheader("📈 Prediction Result")
            if "prediction" in st.session_state:
                pred = st.session_state["prediction"]
                st.success(f"Predicted Segment: **{pred['segment']}**")
                st.info(f"Cluster ID: {pred['cluster']}")
                c1, c2, c3 = st.columns(3)
                c1.metric("Recency", f"{pred['recency']} days")
                c2.metric("Frequency", f"{pred['frequency']} purchases")
                c3.metric("Monetary", f"£{pred['monetary']:,.2f}")
            else:
                st.info("👈 Enter RFM values and click Predict Segment.")

st.markdown("---")
st.caption("🛒 Shopper Spectrum | Customer Segmentation & Product Recommendation")
'''

# Save the Streamlit app code
with open('app/app.py', 'w', encoding='utf-8') as f:
    f.write(streamlit_code)

print("✅ Streamlit app code saved to: app/app.py")
print("\n" + "=" * 70)
print("🚀 AUTO-LAUNCHING STREAMLIT...")
print("=" * 70)

import subprocess
import time
from IPython.display import display, HTML

# Launch Streamlit in background
process = subprocess.Popen(
    ['streamlit', 'run', 'app/app.py', '--server.port=8501'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Wait for server to start
time.sleep(5)

# Display clickable link
display(HTML("""
<div style="padding:25px; background-color:#e8f5e9; border-radius:15px; border:3px solid #4CAF50; text-align:center;">
    <h2 style="margin:0; color:#2e7d32;">🎉 Streamlit App is Running!</h2>
    <p style="font-size:16px; margin-top:10px;">Click the link below to open your app:</p>
    <a href="http://localhost:8501" target="_blank" style="font-size:22px; font-weight:bold; color:#1b5e20;">
        🌐 http://localhost:8501
    </a>
    <p style="margin-top:15px; color:#666; font-size:13px;">
        <b>Note:</b> If the link doesn't open, copy-paste the URL into your browser.<br>
        To stop the app, click the ⏹️ Stop button on this cell or restart the kernel.
    </p>
</div>
"""))

print("\n📡 Streamlit server logs (press Stop to halt):")
print("-" * 50)

# Keep cell alive and print logs
try:
    while True:
        output = process.stdout.readline()
        if output:
            print(output.strip())
        if process.poll() is not None:
            break
except KeyboardInterrupt:
    print("\n🛑 Stopping Streamlit...")
    process.terminate()
    process.wait()
    print("✅ Streamlit stopped successfully.")

print("\n🎉 STEP 6 COMPLETE! Project is ready.")

---

## ✅ Project Completion Summary

| Step | Task | Output | Status |
|------|------|--------|--------|
| 1 | Environment Setup | Folders, venv, packages | ✅ |
| 2 | Data Preprocessing | `online_retail_cleaned.csv` | ✅ |
| 3 | EDA | 12 charts + 5 CSV summaries | ✅ |
| 4 | RFM & Segmentation | KMeans model, scaler, RFM table, cluster profiles | ✅ |
| 5 | Product Recommendation | Similarity matrix, product list | ✅ |
| 6 | Streamlit App | `app.py` auto-generated & launched | ✅ |

### 📁 Final Deliverables

- **`notebooks/shopper_spectrum.ipynb`** — This notebook (complete analysis)
- **`app/app.py`** — Streamlit web application
- **`models/`** — Saved models (kmeans, scaler, similarity matrix, product list)
- **`outputs/`** — CSV summaries, cluster profiles, RFM table, EDA charts
- **`data/online_retail_cleaned.csv`** — Cleaned dataset

### 🎯 Business Insights

- **High-Value customers** are recent, frequent, high-spending — prioritize loyalty programs
- **Regular customers** are steady — target with upsell/cross-sell campaigns
- **Occasional customers** buy rarely — engage with discounts and promotions
- **At-Risk customers** haven't purchased recently — launch win-back campaigns

### 🛠 Technical Stack Used

`Pandas`, `NumPy`, `Matplotlib`, `Seaborn`, `Scikit-learn` (KMeans, StandardScaler, Silhouette, CosineSimilarity), `Streamlit`, `Joblib`

---

**Project completed within 8-day timeline.** 🎉